# Case 4 — Geospatial Analytics untuk Digital Bank (Snowflake Notebook)

**Amar Bank — Workshop Part 2**

Use case: **Location Intelligence & Geo-Fraud** untuk digital bank.
- **A. Coverage** — seberapa jauh nasabah dari titik layanan (ATM/Cabang/Agen); temukan nasabah *underserved* & usulan lokasi baru.
- **B. Geo-Fraud** — deteksi *fraud ring* (banyak aplikasi dari titik sama) & *impossible travel* (satu nasabah, dua aplikasi berjauhan dalam waktu singkat).

Fungsi Snowflake native: `ST_MAKEPOINT`, `ST_DISTANCE`, `ST_CENTROID`, `H3_LATLNG_TO_CELL`, dll — tanpa library eksternal untuk komputasi. Visualisasi peta pakai **plotly** (`scatter_geo`, offline) - pydeck tak dipakai karena Snowflake Notebook memblokir <script> di output HTML.

**Tabel:** `CUSTOMERS_GEO` (10k), `SERVICE_POINTS` (200), `LOAN_APPLICATIONS_GEO` (15k).

**Runtime:** Notebook on **Container Runtime / SPCS** - tidak ada tombol Packages. Paket eksternal di-install di cell pertama: `!pip install pydeck plotly` (pandas/numpy/streamlit sudah tersedia). Bila akun air-gapped, pra-instal ke image.

In [ ]:
# Notebook on Container Runtime / SPCS: install paket yang belum pra-instal (BUKAN via tombol Packages).
# Butuh PyPI External Access Integration aktif di notebook (lihat README case4).
!pip install plotly

In [ ]:
# Import & Snowpark session
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database('AMAR_WORKSHOP_P2')
session.use_schema('GEO')
print('Session OK ->', session.get_current_database(), session.get_current_schema())

# Catatan: pydeck TIDAK dipakai. Snowflake Notebook memblokir <script> di output HTML,
# jadi peta dirender dengan PLOTLY scatter_geo (offline, tanpa token) yang didukung native.

## 1. Muat data geospatial
Ambil 3 tabel ke pandas. `latitude`/`longitude` adalah titik lokasi (derajat desimal).

In [ ]:
cust = session.table('CUSTOMERS_GEO').to_pandas()
sp   = session.table('SERVICE_POINTS').to_pandas()
apps = session.table('LOAN_APPLICATIONS_GEO').to_pandas()
print('customers', cust.shape, '| service_points', sp.shape, '| applications', apps.shape)
cust.head()

## 2. Membuat titik GEOGRAPHY — `ST_MAKEPOINT`
`ST_MAKEPOINT(longitude, latitude)` membentuk objek GEOGRAPHY (titik di bumi).
**Perhatikan urutan: longitude dulu, baru latitude.** Titik ini dasar semua analisa jarak.

In [ ]:
demo = session.sql('''
  SELECT customer_id, city,
         ST_ASWKT(ST_MAKEPOINT(longitude, latitude)) AS geo_point
  FROM CUSTOMERS_GEO LIMIT 5
''').to_pandas()
demo

## 3. Coverage — jarak nasabah ke titik layanan TERDEKAT (`ST_DISTANCE`)
`ST_DISTANCE(g1, g2)` = jarak geodesik (meter) antara dua titik.
Kita CROSS JOIN nasabah × service point, ambil **MIN** jarak per nasabah = jarak ke layanan terdekat. Ini menjawab *"titik A (nasabah) ke titik B (layanan) berapa jauh?"*.

In [ ]:
coverage = session.sql('''
  WITH c AS (SELECT customer_id, city, latitude, longitude,
                    ST_MAKEPOINT(longitude, latitude) g FROM CUSTOMERS_GEO),
       s AS (SELECT service_point_id, ST_MAKEPOINT(longitude, latitude) g FROM SERVICE_POINTS)
  SELECT c.customer_id, c.city, c.latitude, c.longitude,
         MIN(ST_DISTANCE(c.g, s.g))/1000 AS nearest_km
  FROM c CROSS JOIN s
  GROUP BY 1,2,3,4
''').to_pandas()
coverage['UNDERSERVED'] = (coverage['NEAREST_KM'] > 5).astype(int)
print('Rata2 jarak ke layanan terdekat (km):', round(coverage.NEAREST_KM.mean(),2))
print('Nasabah underserved (>5km):', int(coverage.UNDERSERVED.sum()))
coverage.head()

### 3a. Coverage per kota
Kota dengan rata-rata jarak tinggi = kandidat penambahan ATM/cabang.

In [ ]:
import plotly.express as px
bycity = coverage.groupby('CITY', as_index=False)['NEAREST_KM'].mean().sort_values('NEAREST_KM', ascending=False)
px.bar(bycity, x='CITY', y='NEAREST_KM', color='NEAREST_KM', color_continuous_scale='Reds',
       title='Rata-rata jarak ke layanan terdekat per kota (km)')

## 4. Visualisasi peta — nasabah & titik layanan (plotly)
Titik nasabah diwarnai: **merah = underserved (>5km)**, **biru = tercover**. Titik hijau = service point.
Ini memvisualisasikan lokasi titik A (nasabah) dan titik B (layanan).

In [ ]:
c = coverage.copy()
c['STATUS'] = c['UNDERSERVED'].map({1:'Underserved (>5km)', 0:'Tercover'})
fig = px.scatter_geo(c, lat='LATITUDE', lon='LONGITUDE', color='STATUS',
                     color_discrete_map={'Underserved (>5km)':'#dc3232','Tercover':'#1e78c8'},
                     hover_name='CITY', hover_data={'NEAREST_KM':':.1f','LATITUDE':False,'LONGITUDE':False},
                     title='Nasabah & titik layanan (merah = underserved)')
fig.add_trace(go.Scattergeo(lat=sp['LATITUDE'], lon=sp['LONGITUDE'], mode='markers',
              marker=dict(size=8, color='#14b45a', symbol='triangle-up'), name='Service Point'))
fig.update_geos(fitbounds='locations', showcountries=True, resolution=50)
fig.update_layout(height=520, legend_title_text='')
fig

## 5. Garis titik A → titik B (nasabah → layanan terdekat)
Untuk sampel nasabah underserved, kita tarik **garis** ke layanan terdekat memakai plotly `Scattergeo` (mode lines) — memperjelas jarak yang harus ditempuh.

In [ ]:
sample = session.sql('''
  WITH c AS (SELECT customer_id, latitude clat, longitude clon, ST_MAKEPOINT(longitude,latitude) g FROM CUSTOMERS_GEO),
       s AS (SELECT service_point_id, latitude slat, longitude slon, ST_MAKEPOINT(longitude,latitude) g FROM SERVICE_POINTS)
  SELECT customer_id, clat, clon, slat, slon, km FROM (
    SELECT c.customer_id, c.clat, c.clon, s.slat, s.slon,
           ST_DISTANCE(c.g,s.g)/1000 km,
           ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY ST_DISTANCE(c.g,s.g)) rn
    FROM c CROSS JOIN s )
  WHERE rn=1 AND km > 5
  ORDER BY km DESC LIMIT 200
''').to_pandas()

# Garis nasabah -> layanan terdekat: satu trace dengan pemisah None antar segmen
lat, lon = [], []
for _, r in sample.iterrows():
    lat += [r.CLAT, r.SLAT, None]
    lon += [r.CLON, r.SLON, None]
fig = go.Figure()
fig.add_trace(go.Scattergeo(lat=lat, lon=lon, mode='lines',
              line=dict(width=1, color='#dc3232'), name='rute'))
fig.add_trace(go.Scattergeo(lat=sample.CLAT, lon=sample.CLON, mode='markers',
              marker=dict(size=4, color='#dc3232'), name='Nasabah underserved'))
fig.update_geos(fitbounds='locations', showcountries=True, resolution=50)
fig.update_layout(height=520, title='Nasabah underserved -> layanan terdekat (>5km)')
fig

## 6. H3 Hexagon — sebaran & risiko default per area
**H3** membagi bumi jadi hexagon. `H3_LATLNG_TO_CELL(lat,lon,res)` memberi ID sel.
Kita agregasi jumlah nasabah & default rate per hexagon → heatmap risiko wilayah.

In [ ]:
h3 = session.sql('''
  WITH agg AS (
    SELECT H3_LATLNG_TO_CELL(latitude, longitude, 7) AS cell,
           COUNT(*) n_customers, ROUND(AVG(default_flag)*100,1) default_rate
    FROM CUSTOMERS_GEO GROUP BY 1)
  SELECT n_customers, default_rate,
         ST_Y(H3_CELL_TO_POINT(cell)) AS lat,
         ST_X(H3_CELL_TO_POINT(cell)) AS lon
  FROM agg
''').to_pandas()
print('jumlah hexagon:', len(h3))
fig = px.scatter_geo(h3, lat='LAT', lon='LON', size='N_CUSTOMERS', color='DEFAULT_RATE',
                     color_continuous_scale='YlOrRd',
                     hover_data={'N_CUSTOMERS':True,'DEFAULT_RATE':True,'LAT':False,'LON':False},
                     title='Sebaran nasabah & default rate per area (H3 res 7, titik = centroid hexagon)')
fig.update_geos(fitbounds='locations', showcountries=True, resolution=50)
fig.update_layout(height=520)
fig

## 7. Geo-Fraud (A) — *fraud ring*: banyak aplikasi dari titik sama
Aplikasi sah tersebar; **cluster titik nyaris identik** mencurigakan. Kita pakai H3 resolusi tinggi (res 9 ≈ 174m) dan hitung aplikasi per sel. Sel dengan aplikasi jauh di atas normal = kandidat fraud.

In [ ]:
cluster = session.sql('''
  SELECT H3_LATLNG_TO_CELL_STRING(latitude, longitude, 9) h3,
         COUNT(*) n_apps, COUNT(DISTINCT customer_id) n_cust
  FROM LOAN_APPLICATIONS_GEO GROUP BY 1
  ORDER BY n_apps DESC
''').to_pandas()
print('Top cluster (kandidat fraud ring):')
display(cluster.head(5))
thr = cluster.N_APPS.quantile(0.999)
print('Ambang p99.9 =', round(thr,1), '-> sel di atas ambang:', int((cluster.N_APPS>thr).sum()))

## 8. Geo-Fraud (B) — *impossible travel*
Satu nasabah mengajukan dari dua lokasi **berjauhan (>100 km)** dalam **selang < 3 jam** — mustahil ditempuh → indikasi akun disalahgunakan. `ST_DISTANCE` + `LAG` window.
> Catatan: `LAG` tidak menerima GEOGRAPHY, jadi kita LAG lat/long lalu bentuk titik.

In [ ]:
travel = session.sql('''
  WITH a AS (
    SELECT customer_id, submit_ts::timestamp ts, latitude, longitude,
           LAG(submit_ts::timestamp) OVER (PARTITION BY customer_id ORDER BY submit_ts) prev_ts,
           LAG(latitude)  OVER (PARTITION BY customer_id ORDER BY submit_ts) prev_lat,
           LAG(longitude) OVER (PARTITION BY customer_id ORDER BY submit_ts) prev_lon
    FROM LOAN_APPLICATIONS_GEO)
  SELECT customer_id, prev_ts, ts,
         ROUND(ST_DISTANCE(ST_MAKEPOINT(longitude,latitude), ST_MAKEPOINT(prev_lon,prev_lat))/1000,1) dist_km,
         DATEDIFF('minute', prev_ts, ts) mins,
         latitude, longitude, prev_lat, prev_lon
  FROM a
  WHERE prev_ts IS NOT NULL
    AND ST_DISTANCE(ST_MAKEPOINT(longitude,latitude), ST_MAKEPOINT(prev_lon,prev_lat))/1000 > 100
    AND DATEDIFF('minute', prev_ts, ts) < 180
''').to_pandas()
print('Kasus impossible travel terdeteksi:', len(travel))
travel.head()

In [ ]:
if len(travel):
    lat, lon = [], []
    for _, r in travel.iterrows():
        lat += [r.PREV_LAT, r.LATITUDE, None]
        lon += [r.PREV_LON, r.LONGITUDE, None]
    fig = go.Figure(go.Scattergeo(lat=lat, lon=lon, mode='lines+markers',
                    line=dict(width=2, color='#e61e1e'), marker=dict(size=3, color='#e61e1e')))
    fig.update_geos(fitbounds='locations', showcountries=True, resolution=50)
    fig.update_layout(height=520, title='Impossible travel (>100km dalam <180 menit)')
    display(fig)
else:
    print('Tidak ada kasus (coba longgarkan ambang).')

## 9. Simpan hasil ke Snowflake
Simpan flag underserved & kasus fraud untuk downstream (dashboard / alert).

In [ ]:
session.write_pandas(coverage, 'CUSTOMER_COVERAGE', auto_create_table=True, overwrite=True)
if len(travel):
    session.write_pandas(travel, 'GEO_FRAUD_IMPOSSIBLE_TRAVEL', auto_create_table=True, overwrite=True)
print('Tersimpan: CUSTOMER_COVERAGE', len(coverage), '| impossible travel', len(travel))

## 10. Kesimpulan
- **Coverage:** identifikasi nasabah jauh dari layanan → optimasi jaringan ATM/cabang/agen.
- **H3 risk map:** wilayah dengan konsentrasi & default tinggi.
- **Geo-fraud:** *fraud ring* (cluster titik) & *impossible travel* (jarak vs waktu).

Semua memakai fungsi geospatial **native Snowflake** (`ST_*`, `H3_*`) + visualisasi **pydeck**.

✅ Selesai — Case 4.